# Trustworthy Anomaly Agent on ESA-ADB — end-to-end walkthrough

This notebook runs the **complete pipeline** of the project on real satellite telemetry
(ESA's Mission2, lightweight subset): it detects anomalies, calibrates how much you can
trust each detection, explains which channels caused it, and generates an operator alert
whose text is audited by an LLM judge before it reaches you. Along the way it explains
the **key design decisions** of each module: not just what runs, but *why it is built
this way*. For the project context (the operator's problem, the dataset, the
trustworthiness gap) see the [README](../README.md).

```
Telemetry (Mission2, channels 18-28)
  → [1] Detection      — Windowed Isolation Forest        → anomaly score per window
  → [2] Uncertainty    — conformal prediction             → calibrated confidence
  → [3] Attribution    — channel-level explanation        → which sensors caused it
  → [4] Alert layer    — grounded LLM brief + judge       → auditable operator alert
  → [5] This notebook  — the whole chain, live
```

**How to use this notebook.** Reading it on GitHub (outputs are committed) takes
~15-20 min and requires nothing. Running it yourself requires downloading the dataset
once (~3.8 GB, instructions below); after the one-time preprocessing (~30 min) every
later run skips the heavy steps automatically.

---
**Data attribution.** ESA Anomaly Dataset (ESA-ADB) — © European Space Agency (ESOC),
KP Labs, Airbus Defence and Space — licensed [CC BY 3.0 IGO](https://creativecommons.org/licenses/by/3.0/igo/).
Dataset: [10.5281/zenodo.12528696](https://doi.org/10.5281/zenodo.12528696) ·
Paper: [arXiv:2406.17826](https://arxiv.org/abs/2406.17826) ·
Code: [kplabs-pl/ESA-ADB](https://github.com/kplabs-pl/ESA-ADB) (MIT), minimally
vendored under `src/` (see `NOTICE`). All results below are traceable to the dataset —
no invented data.


## 0 · Setup & data

The pipeline needs the raw Mission2 telemetry **once**. If you only want to read this
notebook, skip ahead — every cell below is already executed.

1. Download the **ESA-Mission2** folder (~3.8 GB) from Zenodo:
   [doi.org/10.5281/zenodo.12528696](https://doi.org/10.5281/zenodo.12528696)
2. Place it at `data/ESA-Mission2/` (so that `data/ESA-Mission2/labels.csv` exists).
3. Run the cells below. The first run preprocesses the raw channels into the
   train/test CSVs (~30 min, one-time); later runs detect the existing output and skip.

Everything is deterministic: same input, same bytes out — down to the final
F0.5 = 0.9487. There is no hidden state and no cached shortcut in the pipeline itself.


In [1]:
import sys, time, subprocess
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # notebook lives in notebooks/
import config

TIMINGS: dict = {}  # wall-clock per heavy step, summarized at the end

RAW_DIR = config.REPO / "data" / "ESA-Mission2"
preprocessed = config.TEST_CSV.exists() and config.TRAIN_CSV.exists()

if preprocessed:
    print(f"[ok] Preprocessed CSVs found — ready.\n     {config.TEST_CSV.name}, {config.TRAIN_CSV.name}")
elif RAW_DIR.exists():
    print("[ok] Raw dataset found — the next cell will preprocess it (~30 min, one-time).")
else:
    raise SystemExit(
        "Dataset not found.\n"
        f"  Expected raw data at:        {RAW_DIR}\n"
        f"  (or preprocessed CSVs at:    {config.PREP_DIR})\n"
        "  Download ESA-Mission2 (~3.8 GB): https://doi.org/10.5281/zenodo.12528696\n"
        "  Then re-run this cell. See section '0 · Setup & data' above."
    )


[ok] Preprocessed CSVs found — ready.
     21_months.test.csv, 21_months.train.csv


In [2]:
# [1a] Raw telemetry -> canonical train/test CSVs (ESA's own prep, vendored — see NOTICE).
# Deterministic: depends ONLY on the immutable Zenodo raw data + preprocessing.py.
# To force regeneration (e.g. after editing the script): delete data/preprocessed_subset/.
if preprocessed:
    print("[skip] Already preprocessed — ~30 min saved.")
else:
    print("[run ] Preprocessing channels 18-28 (~30 min on Apple Silicon)...")
    t0 = time.perf_counter()
    subprocess.run(
        [sys.executable, str(config.REPO / "src" / "m1_detection" / "preprocessing.py"), str(RAW_DIR)],
        check=True,
    )
    TIMINGS["preprocess"] = time.perf_counter() - t0
    print(f"[done] Preprocessing finished in {TIMINGS['preprocess']/60:.1f} min.")


[skip] Already preprocessed — ~30 min saved.


## 1 · Detection — Windowed Isolation Forest

*(section coming in Fase 2)*

## 2 · Uncertainty — conformal prediction

*(section coming in Fase 3)*

## 3 · Attribution — which channels caused it

*(section coming in Fase 4)*

## 4 · Trustworthy alert — grounded brief + LLM judge

*(section coming in Fase 6)*

## 5 · Live streaming simulation

*(section coming in Fase 7)*

## 6 · Wrap-up — what this is *not*, future work, references

*(section coming in Fase 8)*